# Percolation scan (PBM)

This notebook builds a 3D lattice input DB, runs percolation scans, and plots the resulting percolation curves.

Folder convention:
- repo root contains `pbm/` and `cli/`; SQL schemas are in `pbm/resources/sql/`
- this notebook lives in `notebooks/`


## Initialize
- Load libraries
- Define repository

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import sqlite3
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
PERC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {PERC_WORKERS} percolation workers.")

# Notebook is expected at: <repo_root>/notebooks/percolation.ipynb
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_percolation.db'
OUT_DB = REPO_ROOT / 'output' / 'results_percolation.db'
FIGS   = REPO_ROOT / 'output' / 'figs_percolation'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

# Plot rendering: 'standard' for portable previews, 'publication' for external-LaTeX PDF export.
PLOT_MODE = 'standard'

from pbm.network.builder import build_and_insert
import cli.cli_simulation as cli_sim

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)


## Percolation scan 1

In [ ]:
# --- [1] Parameters  ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 1e-2

SAMPLES = 10_000_000
THRESHOLD = 2
SEED = 123

# Percolation scan specs
PERCOLATION_SCAN = 'xyz:0,1,0.025' #'xyz:0,1,0.05'   # x=y=z=t

LOG_ATTEMPTS = 'no'  # 'yes' or 'no'

# --- [2] Build lattice input DB ---
build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run simulation + percolation scan ---
print('Simulation running...')
sys.argv = [
    'cli_simulation.py',
    '--db', str(IN_DB),
    '--outdb', str(OUT_DB),
    '--samples', str(SAMPLES),
    '--threshold', str(THRESHOLD),
    '--seed', str(SEED),
    '--log-attempts', str(LOG_ATTEMPTS),
    '--percolation-scan', str(PERCOLATION_SCAN),
    '--percolation-workers', str(PERC_WORKERS),
]

cli_sim.main()

print('Simulation finished:', OUT_DB)

# --- [5] Post-process plots (writes files under output/figs) ---
import cli.cli_post_processing as cli_post

sys.argv = [
    'cli_post_processing.py',
    '--outdb', str(OUT_DB),
    '--figs-root', str(FIGS),
    '--make-plots', 'yes',
    '--plot-mode', PLOT_MODE,
]

cli_post.main()
print('Figures written to:', FIGS)

## Percolation scan 2

In [ ]:
# --- [1] Parameters  ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.8

SAMPLES = 10_000_000
THRESHOLD = 2
SEED = 123

# Percolation scan specs
PERCOLATION_SCAN = 'xy:0,1,0.025'   # xy=t, z=0.8

LOG_ATTEMPTS = 'no'  # 'yes' or 'no'

# --- [2] Build lattice input DB ---
build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run simulation + percolation scan ---
print('Simulation running...')
sys.argv = [
    'cli_simulation.py',
    '--db', str(IN_DB),
    '--outdb', str(OUT_DB),
    '--samples', str(SAMPLES),
    '--threshold', str(THRESHOLD),
    '--seed', str(SEED),
    '--log-attempts', str(LOG_ATTEMPTS),
    '--percolation-scan', str(PERCOLATION_SCAN),
    '--percolation-workers', str(PERC_WORKERS),
]

cli_sim.main()

print('Simulation finished:', OUT_DB)

# --- [5] Post-process plots (writes files under output/figs) ---
import cli.cli_post_processing as cli_post

sys.argv = [
    'cli_post_processing.py',
    '--outdb', str(OUT_DB),
    '--figs-root', str(FIGS),
    '--make-plots', 'yes',
    '--plot-mode', PLOT_MODE,
]

cli_post.main()
print('Figures written to:', FIGS)

## Percolation scan 3

In [ ]:
# --- [1] Parameters  ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.8

SAMPLES = 10_000_000
THRESHOLD = 2
SEED = 123

# Percolation scan specs
PERCOLATION_SCAN = 'z:0,1,0.01'   # z=t, xy=0.8

LOG_ATTEMPTS = 'no'  # 'yes' or 'no'

# --- [2] Build lattice input DB ---
build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run simulation + percolation scan ---
print('Simulation running...')
sys.argv = [
    'cli_simulation.py',
    '--db', str(IN_DB),
    '--outdb', str(OUT_DB),
    '--samples', str(SAMPLES),
    '--threshold', str(THRESHOLD),
    '--seed', str(SEED),
    '--log-attempts', str(LOG_ATTEMPTS),
    '--percolation-scan', str(PERCOLATION_SCAN),
    '--percolation-workers', str(PERC_WORKERS),
]

cli_sim.main()

print('Simulation finished:', OUT_DB)

# --- [5] Post-process plots (writes files under output/figs) ---
import cli.cli_post_processing as cli_post

sys.argv = [
    'cli_post_processing.py',
    '--outdb', str(OUT_DB),
    '--figs-root', str(FIGS),
    '--make-plots', 'yes',
    '--plot-mode', PLOT_MODE,
]

cli_post.main()
print('Figures written to:', FIGS)

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import sqlite3
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
PERC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {PERC_WORKERS} percolation workers.")

# Notebook is expected at: <repo_root>/notebooks/percolation.ipynb
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_percolation.db'
OUT_DB = REPO_ROOT / 'output' / 'results_percolation.db'
FIGS   = REPO_ROOT / 'output' / 'figs_percolation'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

# Plot rendering: 'standard' for portable previews, 'publication' for external-LaTeX PDF export.
PLOT_MODE = 'standard'

from pbm.network.builder import build_and_insert
import cli.cli_simulation as cli_sim

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [5] Post-process plots (writes files under output/figs) ---
import cli.cli_post_processing as cli_post

sys.argv = [
    'cli_post_processing.py',
    '--outdb', str(OUT_DB),
    '--figs-root', str(FIGS),
    '--make-plots', 'yes',
    '--plot-mode', PLOT_MODE,
    '--run-id', '2',
]

cli_post.main()
print('Figures written to:', FIGS)